In [1]:
import os

# Quay về thư mục gốc của Kaggle
%cd /kaggle/working

if not os.path.exists('/kaggle/working/OW-DETR'):
    !git clone https://github.com/nta2112/OW-DETR.git /kaggle/working/OW-DETR
else:
    print("Thư mục OW-DETR đã tồn tại. Đang đồng bộ cập nhật mới nhất từ GitHub...")
    %cd /kaggle/working/OW-DETR
    !git pull origin main

# Chuyển hẳn vào thư mục project
%cd /kaggle/working/OW-DETR


/kaggle/working
Cloning into '/kaggle/working/OW-DETR'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 211 (delta 83), reused 207 (delta 79), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 1.14 MiB | 6.22 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/kaggle/working/OW-DETR


In [2]:
%cd /kaggle/working/OW-DETR

# 1. Cài đặt các thư viện
!pip install -r requirements.txt

# 2. Tải mô hình xương sống DINO ResNet-50
!mkdir -p models
!wget -N https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth -O models/dino_resnet50_pretrain.pth

# 3. Biên dịch toán tử CUDA
%cd models/ops
!sh make.sh
# Kiểm tra biên dịch (Nếu in ra toàn bộ True là thành công)
!python test.py

%cd /kaggle/working/OW-DETR


/kaggle/working/OW-DETR
for details.

--2026-06-30 17:05:18--  https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.180.116, 65.9.180.48, 65.9.180.50, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.180.116|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 94345885 (90M) [application/zip]
Saving to: ‘models/dino_resnet50_pretrain.pth’

models/dino_resnet5 100%[===================>]  89.97M   347MB/s    in 0.3s    

2026-06-30 17:05:18 (347 MB/s) - ‘models/dino_resnet50_pretrain.pth’ saved [94345885/94345885]

/kaggle/working/OW-DETR/models/ops
running build
running build_py
creating build/lib.linux-x86_64-cpython-312/modules
copying modules/__init__.py -> build/lib.linux-x86_64-cpython-312/modules
copying modules/ms_deform_attn.py -> build/lib.linux-x86_64-cpython-312/modules
creating build/lib.linux-x86_64-cpython-312/functions
copying funct

In [3]:
%cd /kaggle/working/OW-DETR

!python tools/convert_ip102_to_voc.py \
    --coco_dir /kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations \
    --img_dir /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification \
    --classes_file /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classes.txt \
    --output_dir data/IP102


/kaggle/working/OW-DETR
Đã copy /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classes.txt sang data/IP102/VOC2007/classes.txt
Đang xử lý split: train từ train.json...
loading annotations into memory...
Done (t=0.26s)
creating index...
index created!
Đã xử lý 2000 ảnh...
Đã xử lý 4000 ảnh...
Đã xử lý 6000 ảnh...
Đã xử lý 8000 ảnh...
Đã xử lý 10000 ảnh...
Đã xử lý 12000 ảnh...
Đã xử lý 14000 ảnh...
Đã xử lý 16000 ảnh...
Đã xử lý 18000 ảnh...
Đã xử lý 20000 ảnh...
Đã xử lý 22000 ảnh...
Đã xử lý 24000 ảnh...
Đã xử lý 26000 ảnh...
Đã xử lý 28000 ảnh...
Đã xử lý 30000 ảnh...
Đã xử lý 32000 ảnh...
Đã xử lý 34000 ảnh...
Đã xử lý 36000 ảnh...
Đã xử lý 38000 ảnh...
Đã xử lý 40000 ảnh...
Đã xử lý 42000 ảnh...
Đã xử lý 44000 ảnh...
Đã hoàn thành split: train.
Đang xử lý split: val từ val.json...
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Đã xử lý 2000 ảnh...
Đã xử lý 4000 ảnh...
Đã xử lý 6000 ảnh...
Đã hoàn thành split: val.
Đang xử lý split: test từ test.jso

In [4]:
%cd /kaggle/working/OW-DETR

from pycocotools.coco import COCO
import os

json_path = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations/train.json"
coco = COCO(json_path)

# Định nghĩa chia lớp theo yêu cầu mới:
# Task 1: 27 lớp (0 - 26)
# Task 2: 25 lớp (27 - 51)
# Task 3: 25 lớp (52 - 76)
# Task 4: 25 lớp (77 - 101)
task_splits = {
    't1': list(range(0, 27)),
    't2': list(range(27, 52)),
    't3': list(range(52, 77)),
    't4': list(range(77, 102))
}

# Tạo tập train cho các Task
train_images = {}
for task_name, classes in task_splits.items():
    task_img_ids = set()
    for ann_id, ann in coco.anns.items():
        if ann['category_id'] in classes:
            task_img_ids.add(ann['image_id'])
    train_images[task_name] = sorted(list(task_img_ids))

# Ghi tất cả các file splits ra đĩa
output_dir = "data/IP102/VOC2007/ImageSets/Main"
os.makedirs(output_dir, exist_ok=True)

splits = {
    't1_train': train_images['t1'],
    't2_train': train_images['t2'],
    't3_train': train_images['t3'],
    't4_train': train_images['t4']
}

for name, img_ids in splits.items():
    path = os.path.join(output_dir, f"{name}.txt")
    with open(path, "w") as f:
        for img_id in img_ids:
            f.write(f"{img_id:06d}\n")
    print(f"Đã tạo {name}.txt với {len(img_ids)} ảnh.")


/kaggle/working/OW-DETR
loading annotations into memory...
Done (t=0.17s)
creating index...
index created!
Đã tạo thành công file t1_train.txt với 12694 bức ảnh chứa 25 lớp đầu tiên.


In [6]:
%cd /kaggle/working/OW-DETR

!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t1' \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 27 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 9 \
    --epochs 10



/kaggle/working/OW-DETR
W0630 17:11:26.002000 122 torch/distributed/run.py:852] 
W0630 17:11:26.002000 122 torch/distributed/run.py:852] *****************************************
W0630 17:11:26.002000 122 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0630 17:11:26.002000 122 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'spo

In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 0 \
    --CUR_INTRODUCED_CLS 27 \
    --data_root 'data/IP102' \
    --train_set 't1_train' \
    --val_set 'test' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t1/checkpoint.pth' \
    --eval


W0630 18:27:51.031000 187 torch/distributed/run.py:852] 
W0630 18:27:51.031000 187 torch/distributed/run.py:852] *****************************************
W0630 18:27:51.031000 187 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0630 18:27:51.031000 187 torch/distributed/run.py:852] *****************************************
('aeroplane', 'bicycle', 'bird', 'boat', 'bus', 'car', 'cat', 'cow', 'dog', 'horse', 'motorbike', 'sheep', 'train', 'elephant', 'bear', 'zebra', 'giraffe', 'truck', 'person', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'diningtable', 'pottedplant', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'bed', 'toilet', 'sofa', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'base

## TASK 2: Học thêm lớp 25 - 49 (Incremental Learning)

In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t2' \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 27 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't2_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain '/kaggle/working/exps/ip102_t1/checkpoint.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 9 \
    --epochs 10


In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 27 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't2_train' \
    --val_set 'test' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t2/checkpoint.pth' \
    --eval


## TASK 3: Học thêm lớp 50 - 74 (Incremental Learning)

In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t3' \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 52 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't3_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain '/kaggle/working/exps/ip102_t2/checkpoint.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 9 \
    --epochs 10


In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 52 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't3_train' \
    --val_set 'test' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t3/checkpoint.pth' \
    --eval


## TASK 4: Học thêm lớp 75 - 101 (Incremental Learning)

In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --output_dir '/kaggle/working/exps/ip102_t4' \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 77 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't4_train' \
    --val_set 'val' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --pretrain '/kaggle/working/exps/ip102_t3/checkpoint.pth' \
    --num_workers 4 \
    --eval_every 1 \
    --cache_mode \
    --nc_epoch 9 \
    --epochs 10


In [ ]:
!torchrun --nproc_per_node=2 main_open_world.py \
    --dataset owod \
    --num_queries 100 \
    --batch_size 2 \
    --PREV_INTRODUCED_CLS 77 \
    --CUR_INTRODUCED_CLS 25 \
    --data_root 'data/IP102' \
    --train_set 't4_train' \
    --val_set 'test' \
    --test_set 'test' \
    --num_classes 103 \
    --unmatched_boxes \
    --top_unk 5 \
    --featdim 1024 \
    --NC_branch \
    --backbone 'dino_resnet50' \
    --resume '/kaggle/working/exps/ip102_t4/checkpoint.pth' \
    --eval
